In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import os
from sklearn.metrics import classification_report, confusion_matrix
import pickle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
import time

BASE_PATH = os.path.dirname(os.getcwd())
with open(f"{BASE_PATH}/datasets/material/readyDataset.pkl", "rb") as f : 
    df = pickle.load(f)
print("LOADED")

LOADED


### Models to be Used
We will train several machine learning models to perform text classification:

1. **Naive Bayes** – Probabilistic model suitable for text data.
2. **Logistic Regression** – Linear model for binary and multi-class classification.
3. **Support Vector Machine (SVM)** – Effective in high-dimensional spaces.
4. **XGBoost** – Gradient boosting algorithm for high performance.

---

### Feature Extraction Methods
We will use the following techniques to convert text into numerical features:

1. **CountVectorizer** – Counts the occurrences of words in documents.
2. **TF-IDF** – Weights words based on frequency and importance across the corpus.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

labelEncoder = LabelEncoder()
LABEL = labelEncoder.fit_transform(df["label"])
train_data, test_data, train_label, test_label = train_test_split(df["Cleaned_Text"], LABEL, test_size = 0.2, random_state=42, stratify=LABEL)

In [4]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

countVectorizer = CountVectorizer(min_df = 5, decode_error='ignore')
tfidf = TfidfVectorizer(min_df = 5, decode_error='ignore')

train_data_cv = countVectorizer.fit_transform(train_data).toarray()
train_data_tf = tfidf.fit_transform(train_data).toarray()

test_data_cv = countVectorizer.transform(test_data).toarray()
test_data_tf = tfidf.transform(test_data).toarray()

In [5]:
tfidf_ngram = TfidfVectorizer(ngram_range=(1,2) , min_df = 5, decode_error='ignore')

train_data_tf_ngram = tfidf_ngram.fit_transform(train_data).toarray()
test_data_tf_ngram = tfidf_ngram.transform(test_data).toarray()

### 1.a. NaiveBayes - Count Vectorizer

In [11]:
from sklearn.naive_bayes import MultinomialNB

naiveBayes = MultinomialNB()

start = time.time()
naiveBayes.fit(train_data_cv, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = naiveBayes.predict(test_data_cv)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

Train Time : 2.635532855987549
              precision    recall  f1-score   support

           0      0.840     0.916     0.877      1159
           1      0.860     0.901     0.880      1352
           2      0.762     0.643     0.698       328
           3      0.851     0.792     0.820       542
           4      0.786     0.764     0.775       475
           5      0.750     0.396     0.518       144

    accuracy                          0.835      4000
   macro avg      0.808     0.735     0.761      4000
weighted avg      0.832     0.835     0.830      4000



### 1.b. NaiveBayes - Tfidf Vectorizer

In [9]:
naiveBayes = MultinomialNB()

start = time.time()
naiveBayes.fit(train_data_tf, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = naiveBayes.predict(test_data_tf)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

Train Time : 0.12399697303771973
              precision    recall  f1-score   support

           0      0.741     0.943     0.830      1159
           1      0.666     0.973     0.791      1352
           2      0.957     0.137     0.240       328
           3      0.943     0.491     0.646       542
           4      0.886     0.408     0.559       475
           5      1.000     0.007     0.014       144

    accuracy                          0.729      4000
   macro avg      0.865     0.493     0.513      4000
weighted avg      0.787     0.729     0.682      4000



### 1.c. NaiveBayes - Tfidf Vectorizer + NGram

In [10]:
naiveBayes = MultinomialNB()

start = time.time()
naiveBayes.fit(train_data_tf_ngram, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = naiveBayes.predict(test_data_tf_ngram)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits= 3))

Train Time : 0.5830941200256348
              precision    recall  f1-score   support

           0      0.716     0.909     0.801      1159
           1      0.618     0.975     0.757      1352
           2      1.000     0.058     0.110       328
           3      0.942     0.360     0.521       542
           4      0.898     0.316     0.467       475
           5      1.000     0.021     0.041       144

    accuracy                          0.685      4000
   macro avg      0.862     0.440     0.449      4000
weighted avg      0.769     0.685     0.624      4000



### 2.a. Logistic Regression - Count Vectorizer

In [14]:
from sklearn.linear_model import LogisticRegression

logRegression = LogisticRegression()

start = time.time()
logRegression.fit(train_data_cv, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = logRegression.predict(test_data_cv)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

Train Time : 3.1900441646575928
              precision    recall  f1-score   support

           0      0.914     0.931     0.922      1159
           1      0.895     0.926     0.910      1352
           2      0.818     0.784     0.801       328
           3      0.898     0.849     0.873       542
           4      0.846     0.842     0.844       475
           5      0.760     0.639     0.694       144

    accuracy                          0.885      4000
   macro avg      0.855     0.828     0.841      4000
weighted avg      0.884     0.885     0.884      4000



d:\Conda\envs\tensorflow_support\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 2.b. Logistic Regression - Tfidf Vectorizer

In [15]:
logRegression = LogisticRegression()

start = time.time()
logRegression.fit(train_data_tf, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = logRegression.predict(test_data_tf)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

Train Time : 3.2262790203094482
              precision    recall  f1-score   support

           0      0.881     0.940     0.910      1159
           1      0.837     0.956     0.893      1352
           2      0.896     0.655     0.757       328
           3      0.911     0.795     0.849       542
           4      0.850     0.764     0.805       475
           5      0.886     0.486     0.628       144

    accuracy                          0.866      4000
   macro avg      0.877     0.766     0.807      4000
weighted avg      0.868     0.866     0.861      4000



### 2.c. Logistic Regression - Tfidf Vectorizer + NGram

In [16]:
logRegression = LogisticRegression()

start = time.time()
logRegression.fit(train_data_tf_ngram, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = logRegression.predict(test_data_tf_ngram)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

Train Time : 8.054567337036133
              precision    recall  f1-score   support

           0      0.856     0.937     0.895      1159
           1      0.781     0.957     0.860      1352
           2      0.868     0.561     0.681       328
           3      0.923     0.725     0.812       542
           4      0.859     0.693     0.767       475
           5      0.909     0.347     0.503       144

    accuracy                          0.834      4000
   macro avg      0.866     0.703     0.753      4000
weighted avg      0.843     0.834     0.825      4000



d:\Conda\envs\tensorflow_support\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 3.a. SVC - Count Vectorizer

In [17]:
from sklearn.svm import LinearSVC

linearSVC = LinearSVC(random_state=42)

start = time.time()
linearSVC.fit(train_data_cv, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = linearSVC.predict(test_data_cv)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

Train Time : 1.3003907203674316
              precision    recall  f1-score   support

           0      0.917     0.920     0.919      1159
           1      0.898     0.912     0.905      1352
           2      0.786     0.771     0.778       328
           3      0.883     0.862     0.872       542
           4      0.845     0.848     0.847       475
           5      0.723     0.688     0.705       144

    accuracy                          0.880      4000
   macro avg      0.842     0.833     0.838      4000
weighted avg      0.880     0.880     0.880      4000



### 3.b. SVC - Tfidf Vectorizer

In [18]:
linearSVC = LinearSVC(random_state=42)

start = time.time()
linearSVC.fit(train_data_tf, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = linearSVC.predict(test_data_tf)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

Train Time : 0.5461256504058838
              precision    recall  f1-score   support

           0      0.926     0.934     0.930      1159
           1      0.901     0.928     0.914      1352
           2      0.824     0.799     0.811       328
           3      0.901     0.858     0.879       542
           4      0.857     0.857     0.857       475
           5      0.785     0.708     0.745       144

    accuracy                          0.893      4000
   macro avg      0.865     0.847     0.856      4000
weighted avg      0.892     0.893     0.892      4000



### 3.c. SVC - Tfidf Vectorizer + NGram

In [19]:
linearSVC = LinearSVC(random_state=42)

start = time.time()
linearSVC.fit(train_data_tf_ngram, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = linearSVC.predict(test_data_tf_ngram)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

Train Time : 1.4910595417022705
              precision    recall  f1-score   support

           0      0.918     0.941     0.929      1159
           1      0.885     0.929     0.906      1352
           2      0.817     0.762     0.789       328
           3      0.902     0.830     0.865       542
           4      0.867     0.836     0.851       475
           5      0.820     0.729     0.772       144

    accuracy                          0.887      4000
   macro avg      0.868     0.838     0.852      4000
weighted avg      0.886     0.887     0.886      4000



### 4.a. XGBoost - Count Vectorizer

In [20]:
from xgboost import XGBClassifier

xgboost = XGBClassifier(random_state=42)
start = time.time()
xgboost.fit(train_data_cv, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = xgboost.predict(test_data_cv)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

Train Time : 3.5322699546813965
              precision    recall  f1-score   support

           0      0.947     0.911     0.929      1159
           1      0.895     0.901     0.898      1352
           2      0.764     0.851     0.805       328
           3      0.903     0.860     0.881       542
           4      0.838     0.859     0.848       475
           5      0.705     0.764     0.733       144

    accuracy                          0.884      4000
   macro avg      0.842     0.858     0.849      4000
weighted avg      0.887     0.884     0.885      4000



### 4.b. XGBoost - Tfidf Vectorizer 

In [21]:
xgboost = XGBClassifier(random_state=42)
start = time.time()
xgboost.fit(train_data_tf, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = xgboost.predict(test_data_tf)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

Train Time : 12.30891227722168
              precision    recall  f1-score   support

           0      0.942     0.891     0.916      1159
           1      0.866     0.903     0.884      1352
           2      0.766     0.817     0.791       328
           3      0.895     0.845     0.869       542
           4      0.826     0.840     0.833       475
           5      0.743     0.764     0.753       144

    accuracy                          0.872      4000
   macro avg      0.840     0.843     0.841      4000
weighted avg      0.874     0.872     0.873      4000



### 4.c. XGBoost - Tfidf Vectorizer + NGram

In [22]:
xgboost = XGBClassifier(random_state=42)

start = time.time()
xgboost.fit(train_data_tf_ngram, train_label)
print(f"Train Time : {time.time() - start}")

predicted_result = xgboost.predict(test_data_tf_ngram)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

Train Time : 37.329936265945435
              precision    recall  f1-score   support

           0      0.945     0.891     0.917      1159
           1      0.868     0.912     0.890      1352
           2      0.777     0.829     0.802       328
           3      0.898     0.843     0.870       542
           4      0.832     0.844     0.838       475
           5      0.747     0.757     0.752       144

    accuracy                          0.876      4000
   macro avg      0.844     0.846     0.845      4000
weighted avg      0.878     0.876     0.877      4000



### 5. MLP  (TF-IDF Vectorizer)

In [31]:
model = Sequential(name = "prediction_textMLP")
model.add(Input(shape=(train_data_tf.shape[-1],)))
model.add(Dense(units = 32))
model.add(Dropout(rate = 0.2))
model.add(Dense(units = 64))
model.add(Dropout(rate = 0.3))
model.add(Dense(units = 32))
model.add(Dropout(rate = 0.2))
model.add(Dense(units = 6, activation="softmax"))
model.summary()

Model: "prediction_textMLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 32)             │       108,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 113,350 (442.77 KB)

 Trainable params: 113,350 (442.77 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
optimizer = Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    metrics=['accuracy'],
    loss = "sparse_categorical_crossentropy"
)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_label),
    y=train_label
)
# convert ke dictionary untuk Keras
class_weights = {i: w for i, w in enumerate(class_weights)}
print(class_weights)

{0: np.float64(0.5749604714675866), 1: np.float64(0.4930054846860171), 2: np.float64(2.030972328002031), 3: np.float64(1.2305799107829565), 4: np.float64(1.4049877063575693), 5: np.float64(4.63768115942029)}


In [33]:
start = time.time()
model.fit(  
    x=train_data_tf,  
    y=train_label,  
    batch_size=16,  
    epochs=50,  
    callbacks=[EarlyStopping(monitor = "val_loss", patience = 15, restore_best_weights=True)],  
    class_weight=class_weights,
    verbose=1,
)
print(f"Train Time : {time.time() - start}")

Epoch 1/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6911 - loss: 0.9122
Epoch 2/50
  85/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9058 - loss: 0.2645

d:\Conda\envs\tensorflow_support\lib\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9043 - loss: 0.2825
Epoch 3/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9332 - loss: 0.1899
Epoch 4/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9467 - loss: 0.1456
Epoch 5/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9578 - loss: 0.1198
Epoch 6/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9588 - loss: 0.1198
Epoch 7/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9604 - loss: 0.1116
Epoch 8/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9657 - loss: 0.0981
Epoch 9/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9672 - loss: 0.0950
Epoch 10/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9705 - loss: 0.0828
Epoch 11/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9732 - loss: 0.0772
Epoch 12/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9743 - loss: 0.0784
Epoch 13/50
1000/1000 ━━━━━━━━━━━━━━━━━━

In [43]:
predicted_result = model.predict(test_data_tf)
predicted_class = np.argmax(predicted_result, axis=1)
predicted_result = np.int8(predicted_class)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0      0.896     0.858     0.877      1159
           1      0.886     0.835     0.860      1352
           2      0.637     0.817     0.716       328
           3      0.780     0.817     0.798       542
           4      0.817     0.768     0.792       475
           5      0.615     0.764     0.681       144

    accuracy                          0.828      4000
   macro avg      0.772     0.810     0.787      4000
weighted avg      0.836     0.828     0.830      4000



### 6. MLP  (Count Vectorizer)

In [44]:
model = Sequential(name = "prediction_textMLP")
model.add(Input(shape=(train_data_cv.shape[-1],)))
model.add(Dense(units = 32))
model.add(Dropout(rate = 0.2))
model.add(Dense(units = 64))
model.add(Dropout(rate = 0.3))
model.add(Dense(units = 32))
model.add(Dropout(rate = 0.2))
model.add(Dense(units = 6, activation="softmax"))
model.summary()

Model: "prediction_textMLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 32)             │       108,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 113,350 (442.77 KB)

 Trainable params: 113,350 (442.77 KB)

 Non-trainable params: 0 (0.00 B)

In [45]:
optimizer = Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    metrics=['accuracy'],
    loss = "sparse_categorical_crossentropy"
)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_label),
    y=train_label
)
# convert ke dictionary untuk Keras
class_weights = {i: w for i, w in enumerate(class_weights)}
print(class_weights)

{0: np.float64(0.5749604714675866), 1: np.float64(0.4930054846860171), 2: np.float64(2.030972328002031), 3: np.float64(1.2305799107829565), 4: np.float64(1.4049877063575693), 5: np.float64(4.63768115942029)}


In [46]:
start = time.time()
model.fit(  
    x=train_data_cv,  
    y=train_label,  
    batch_size=16,  
    epochs=50,  
    callbacks=[EarlyStopping(monitor = "val_loss", patience = 15, restore_best_weights=True)],  
    class_weight=class_weights,
    verbose=1,
)
print(f"Train Time : {time.time() - start}")

Epoch 1/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6443 - loss: 1.0228
Epoch 2/50
  98/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8857 - loss: 0.3170

d:\Conda\envs\tensorflow_support\lib\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8922 - loss: 0.3095
Epoch 3/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9237 - loss: 0.2168
Epoch 4/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9360 - loss: 0.1719
Epoch 5/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9391 - loss: 0.1596
Epoch 6/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9484 - loss: 0.1369
Epoch 7/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9514 - loss: 0.1288
Epoch 8/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9535 - loss: 0.1273
Epoch 9/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9609 - loss: 0.1045
Epoch 10/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9616 - loss: 0.1083
Epoch 11/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9628 - loss: 0.0999
Epoch 12/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9678 - loss: 0.0836
Epoch 13/50
1000/1000 ━━━━━━━━━━━━━━━━━━

In [47]:
predicted_result = model.predict(test_data_cv)
predicted_class = np.argmax(predicted_result, axis=1)
predicted_result = np.int8(predicted_class)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 979us/step
              precision    recall  f1-score   support

           0      0.888     0.896     0.892      1159
           1      0.901     0.862     0.881      1352
           2      0.725     0.820     0.770       328
           3      0.837     0.825     0.831       542
           4      0.814     0.804     0.809       475
           5      0.665     0.757     0.708       144

    accuracy                          0.853      4000
   macro avg      0.805     0.827     0.815      4000
weighted avg      0.855     0.853     0.853      4000



### 7. MLP  (TF-IDF Vectorizer + NGram)

In [48]:
model = Sequential(name = "prediction_textMLP")
model.add(Input(shape=(train_data_tf_ngram.shape[-1],)))
model.add(Dense(units = 32))
model.add(Dropout(rate = 0.2))
model.add(Dense(units = 64))
model.add(Dropout(rate = 0.3))
model.add(Dense(units = 64))
model.add(Dropout(rate = 0.3))
model.add(Dense(units = 32))
model.add(Dropout(rate = 0.2))
model.add(Dense(units = 6, activation="softmax"))
model.summary()

Model: "prediction_textMLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_16 (Dense)                │ (None, 32)             │       347,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 6)              │           198 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 355,750 (1.36 MB)

 Trainable params: 355,750 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

In [49]:
optimizer = Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    metrics=['accuracy'],
    loss = "sparse_categorical_crossentropy"
)
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_label),
    y=train_label
)
# convert ke dictionary untuk Keras
class_weights = {i: w for i, w in enumerate(class_weights)}
print(class_weights)

{0: np.float64(0.5749604714675866), 1: np.float64(0.4930054846860171), 2: np.float64(2.030972328002031), 3: np.float64(1.2305799107829565), 4: np.float64(1.4049877063575693), 5: np.float64(4.63768115942029)}


In [50]:
model.fit(  
    x=train_data_tf_ngram,  
    y=train_label,  
    batch_size=16,  
    epochs=50,  
    callbacks=[EarlyStopping(monitor = "val_loss", patience = 15, restore_best_weights=True)],  
    class_weight=class_weights,
    verbose=1,
)

Epoch 1/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.6051 - loss: 1.0253
Epoch 2/50
  81/1000 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9217 - loss: 0.2613

d:\Conda\envs\tensorflow_support\lib\site-packages\keras\src\callbacks\early_stopping.py:99: UserWarning: Early stopping conditioned on metric `val_loss` which is not available. Available metrics are: accuracy,loss
  current = self.get_monitor_value(logs)


1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9104 - loss: 0.2676
Epoch 3/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9567 - loss: 0.1318
Epoch 4/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9711 - loss: 0.0953
Epoch 5/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9764 - loss: 0.0848
Epoch 6/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9766 - loss: 0.0764
Epoch 7/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9838 - loss: 0.0608
Epoch 8/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9833 - loss: 0.0576
Epoch 9/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9877 - loss: 0.0501
Epoch 10/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9862 - loss: 0.0541
Epoch 11/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9887 - loss: 0.0456
Epoch 12/50
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9902 - loss: 0.0423
Epoch 13/50
1000/1000 ━━━━━━━━━━━━━━━━━━

In [51]:
predicted_result = model.predict(test_data_tf_ngram)
predicted_class = np.argmax(predicted_result, axis=1)
predicted_result = np.int8(predicted_class)
print(classification_report(y_true = test_label, y_pred = predicted_result, digits = 3))

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
              precision    recall  f1-score   support

           0      0.907     0.870     0.888      1159
           1      0.854     0.888     0.871      1352
           2      0.748     0.671     0.707       328
           3      0.836     0.825     0.830       542
           4      0.772     0.804     0.788       475
           5      0.639     0.701     0.669       144

    accuracy                          0.840      4000
   macro avg      0.793     0.793     0.792      4000
weighted avg      0.841     0.840     0.840      4000

